# Note 05 Deep Dive: Null Space, Nullity and the Four Fundamental Subspaces

Companion notebook for [`05-Null-Space-and-Nullity.md`](../05-Null-Space-and-Nullity.md). Every number quoted in the note's new sections is produced here.

| Part | Topic |
|---|---|
| A | Exact (sympy `rref` / `nullspace`) vs numerical (`scipy.linalg.null_space`) null spaces; rank–nullity checks |
| B | Complete solution of an underdetermined system: x = x_p + N(A) |
| C | The four fundamental subspaces and their orthogonality |
| D | N(XᵀX) = N(X) and non-unique regression weights |
| E | Case study: balancing chemical equations automatically |
| F | Case study: Hamming(7,4) encode / decode with syndromes over GF(2) |
| G | Case study: conservation of flow in a network (cycle space) |
| H | Case study: multicollinearity, VIF and the dummy-variable trap |
| I | Checks for the practice problems in the note |

Figures for the note are produced by [`figures_05.py`](figures_05.py).

In [1]:
import numpy as np
import sympy as sp
import pandas as pd
from scipy.linalg import null_space
from sklearn.linear_model import LinearRegression
np.set_printoptions(precision=4, suppress=True)

## Part A — Exact vs numerical null spaces

In [2]:
def analyse(name, rows):
    A = sp.Matrix(rows)
    R, piv = A.rref()
    m, n = A.shape
    basis = A.nullspace()
    print(f"=== {name}: {m}x{n} ===")
    print("RREF:"); sp.pprint(R)
    print("pivot columns (1-based):", [p + 1 for p in piv])
    print("exact null-space basis:", [list(v) for v in basis])
    print(f"rank = {A.rank()}, nullity = {len(basis)}, rank + nullity = {A.rank() + len(basis)} = n = {n}")
    for v in basis:
        assert A * v == sp.zeros(m, 1)
    N = null_space(np.array(rows, dtype=float))          # orthonormal basis via SVD
    print("scipy null_space (orthonormal columns):\n", N)
    print("A @ N ≈ 0 ?", np.allclose(np.array(rows, float) @ N, 0), "| NᵀN = I ?", np.allclose(N.T @ N, np.eye(N.shape[1])))
    print()
    return A

A1 = analyse("3x4 example", [[1, 2, 0, 3], [2, 4, 1, 8], [3, 6, 1, 11]])
A2 = analyse("4x5 example", [[1, 3, 0, 2, -1], [2, 6, 1, 7, 0], [0, 0, 1, 4, 3], [3, 9, 1, 9, -1]])
A3 = analyse("rank-deficient 3x3", [[1, 2, -1], [2, 4, -2], [-3, -6, 3]])
A5 = analyse("lecture 2x3 (p.55)", [[1, 1, 2], [1, 0, 3]])

=== 3x4 example: 3x4 ===
RREF:
⎡1  2  0  3⎤
⎢          ⎥
⎢0  0  1  2⎥
⎢          ⎥
⎣0  0  0  0⎦
pivot columns (1-based): [1, 3]
exact null-space basis: [[-2, 1, 0, 0], [-3, 0, -2, 1]]
rank = 2, nullity = 2, rank + nullity = 4 = n = 4
scipy null_space (orthonormal columns):
 [[ 0.9211  0.0677]
 [-0.2767 -0.5789]
 [ 0.245  -0.7268]
 [-0.1225  0.3634]]
A @ N ≈ 0 ? True | NᵀN = I ? True

=== 4x5 example: 4x5 ===
RREF:
⎡1  3  0  0  -3⎤
⎢              ⎥
⎢0  0  1  0  -1⎥
⎢              ⎥
⎢0  0  0  1  1 ⎥
⎢              ⎥
⎣0  0  0  0  0 ⎦
pivot columns (1-based): [1, 3, 4]
exact null-space basis: [[-3, 1, 0, 0, 0], [3, 0, 1, -1, 1]]
rank = 3, nullity = 2, rank + nullity = 5 = n = 5
scipy null_space (orthonormal columns):
 [[ 0.9494  0.1471]
 [-0.3139  0.4573]
 [ 0.0026  0.5064]
 [-0.0026 -0.5064]
 [ 0.0026  0.5064]]
A @ N ≈ 0 ? True | NᵀN = I ? True

=== rank-deficient 3x3: 3x3 ===
RREF:
⎡1  2  -1⎤
⎢        ⎥
⎢0  0  0 ⎥
⎢        ⎥
⎣0  0  0 ⎦
pivot columns (1-based): [1]
exact null-space basis:

**Exact vs numerical.** sympy returns the *special solutions* (one per free variable, with a 1 in that free slot), in exact rational arithmetic. `scipy.linalg.null_space` returns an **orthonormal** basis from the SVD (right singular vectors whose singular value is below a tolerance). The two bases differ but span the **same subspace**: check that each scipy column is a combination of the sympy vectors.

In [3]:
S = np.array([[float(x) for x in v] for v in A1.nullspace()]).T   # 4x2 special solutions
N = null_space(np.array(A1, dtype=float))
coef, res, *_ = np.linalg.lstsq(S, N, rcond=None)
print("residual of expressing scipy basis in sympy basis:", np.abs(S @ coef - N).max().round(12))

# Membership test: v in N(A) iff A v = 0
for v in ([-8, 1, -4, 2], [1, 1, 1, 1]):
    print(v, "-> A v =", list(A1 * sp.Matrix(v)))

residual of expressing scipy basis in sympy basis: 0.0
[-8, 1, -4, 2] -> A v = [0, 0, 0]
[1, 1, 1, 1] -> A v = [6, 15, 21]


**Why the tolerance matters.** Real data are noisy, so an *exactly* redundant column becomes *almost* redundant. Numerically, "null" means "singular value below `rcond × σ_max`".

In [4]:
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 3))
X = np.c_[X, X[:, 0] + X[:, 1] + 1e-6 * rng.normal(size=100)]   # 4th column ≈ col1 + col2
print("singular values:", np.linalg.svd(X, compute_uv=False))
print("null_space default rcond :", null_space(X).shape[1], "vectors")
print("null_space rcond=1e-4    :", null_space(X, rcond=1e-4).shape[1], "vector ->", null_space(X, rcond=1e-4).ravel().round(4))

singular values: [18.4242 10.3754  9.4569  0.    ]
null_space default rcond : 0 vectors
null_space rcond=1e-4    : 1 vector -> [-0.5774 -0.5774  0.      0.5774]


## Part B — Complete solution of Ax = b (underdetermined, consistent)

In [5]:
b = sp.Matrix([4, 11, 15])
R, piv = A1.row_join(b).rref()
sp.pprint(R)
xp = sp.Matrix([4, 0, 3, 0])                     # free variables x2 = x4 = 0
n1, n2 = A1.nullspace()
print("A x_p =", list(A1 * xp))
s, t = sp.symbols("s t")
x = xp + s * n1 + t * n2
print("general solution x(s,t) =", list(x))
print("A x(s,t) - b =", list(sp.simplify(A1 * x - b)))
# A b that is NOT consistent: b3 != b1 + b2
print("rank[A|b'] for b'=(4,11,16):", A1.row_join(sp.Matrix([4, 11, 16])).rank(), "vs rank A =", A1.rank())

⎡1  2  0  3  4⎤
⎢             ⎥
⎢0  0  1  2  3⎥
⎢             ⎥
⎣0  0  0  0  0⎦
A x_p = [4, 11, 15]
general solution x(s,t) = [-2*s - 3*t + 4, s, 3 - 2*t, t]
A x(s,t) - b = [0, 0, 0]
rank[A|b'] for b'=(4,11,16): 3 vs rank A = 2


**Minimum-norm solution.** Among all solutions x_p + N(A), the pseudo-inverse picks the unique one lying in the row space C(Aᵀ), i.e. orthogonal to every null-space vector. The left null space gives the consistency test: Ax = b is solvable iff b ⟂ N(Aᵀ).

In [6]:
x_min = A1.pinv() * b
print("min-norm solution A⁺b =", list(x_min), "≈", [round(float(v), 4) for v in x_min])
print("A x_min =", list(A1 * x_min), "| x_min · n1 =", (x_min.T * n1)[0], "| x_min · n2 =", (x_min.T * n2)[0])
y_left = A1.T.nullspace()[0]
print("left null space N(Aᵀ):", list(y_left), "-> condition", list(y_left), "· b = 0, i.e. b3 = b1 + b2")
for bb in ([4, 11, 15], [4, 11, 16]):
    print("b =", bb, " y·b =", (y_left.T * sp.Matrix(bb))[0])

min-norm solution A⁺b = [1/17, 2/17, 9/17, 21/17] ≈ [0.0588, 0.1176, 0.5294, 1.2353]
A x_min = [4, 11, 15] | x_min · n1 = 0 | x_min · n2 = 0
left null space N(Aᵀ): [-1, -1, 1] -> condition [-1, -1, 1] · b = 0, i.e. b3 = b1 + b2
b = [4, 11, 15]  y·b = 0
b = [4, 11, 16]  y·b = 1


## Part C — The four fundamental subspaces

In [7]:
F = sp.Matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
col  = F.columnspace(); row = F.rowspace(); nul = F.nullspace(); lnul = F.T.nullspace()
print("C(A)  basis:", [list(v) for v in col])
print("C(Aᵀ) basis:", [list(v) for v in row], "  (RREF rows:", [list(F.rref()[0].row(i)) for i in range(2)], ")")
print("N(A)  basis:", [list(v) for v in nul])
print("N(Aᵀ) basis:", [list(v) for v in lnul])
r, (m, n) = F.rank(), F.shape
print(f"dims: C(A)={r}, C(Aᵀ)={r}, N(A)={n-r}, N(Aᵀ)={m-r}")
print("row space ⟂ null space :", [ (rv * nv)[0] for rv in row for nv in nul])
print("col space ⟂ left null  :", [ (cv.T * lv)[0] for cv in col for lv in lnul])

C(A)  basis: [[1, 4, 7], [2, 5, 8]]
C(Aᵀ) basis: [[1, 2, 3], [0, -3, -6]]   (RREF rows: [[1, 0, -1], [0, 1, 2]] )
N(A)  basis: [[1, -2, 1]]
N(Aᵀ) basis: [[1, -2, 1]]
dims: C(A)=2, C(Aᵀ)=2, N(A)=1, N(Aᵀ)=1
row space ⟂ null space : [0, 0]
col space ⟂ left null  : [0, 0]


In [8]:
# The same checks on a non-square, non-symmetric matrix (4x5 from Part A)
r, (m, n) = A2.rank(), A2.shape
row, nul, col, lnul = A2.rowspace(), A2.nullspace(), A2.columnspace(), A2.T.nullspace()
print(f"m={m}, n={n}, rank={r}: dim C(A)={len(col)}, dim C(Aᵀ)={len(row)}, dim N(A)={len(nul)}, dim N(Aᵀ)={len(lnul)}")
print("left null space basis:", [list(v) for v in lnul])
print("max |row·null| =", max(abs((a * b_)[0]) for a in row for b_ in nul))
print("max |col·leftnull| =", max(abs((a.T * b_)[0]) for a in col for b_ in lnul))

m=4, n=5, rank=3: dim C(A)=3, dim C(Aᵀ)=3, dim N(A)=2, dim N(Aᵀ)=1
left null space basis: [[-1, -1, 0, 1]]
max |row·null| = 0
max |col·leftnull| = 0


## Part D — N(XᵀX) = N(X): why exact redundancy breaks OLS

In [9]:
X = sp.Matrix([[1, 2, 3], [1, 0, 1], [2, 1, 3], [0, 1, 1]])     # column 3 = column 1 + column 2
G = X.T * X
print("XᵀX ="); sp.pprint(G)
print("det(XᵀX) =", G.det(), "| rank X =", X.rank(), "| rank XᵀX =", G.rank())
print("N(X)   =", [list(v) for v in X.nullspace()])
print("N(XᵀX) =", [list(v) for v in G.nullspace()])

y = sp.Matrix([6, 2, 5, 1])
w1 = sp.Matrix([1, 1, 1])
z  = X.nullspace()[0]
w2 = w1 + 5 * z
print("w1 =", list(w1), " predictions:", list(X * w1))
print("w2 = w1 + 5z =", list(w2), " predictions:", list(X * w2), "  (identical fits, different weights)")
Xn = np.array(X, dtype=float); yn = np.array(y, dtype=float).ravel()
w_pinv = np.linalg.pinv(Xn) @ yn
print("min-norm (pseudo-inverse) solution:", w_pinv.round(4), "| ⟂ to null vector?", np.isclose(w_pinv @ np.array([-1, -1, 1.]), 0))
lam = 0.1
print("ridge: XᵀX + 0.1 I invertible? det =", round(np.linalg.det(Xn.T @ Xn + lam * np.eye(3)), 4))

XᵀX =
⎡6   4   10⎤
⎢          ⎥
⎢4   6   10⎥
⎢          ⎥
⎣10  10  20⎦
det(XᵀX) = 0 | rank X = 2 | rank XᵀX = 2
N(X)   = [[-1, -1, 1]]
N(XᵀX) = [[-1, -1, 1]]
w1 = [1, 1, 1]  predictions: [6, 2, 6, 2]
w2 = w1 + 5z = [-4, -4, 6]  predictions: [6, 2, 6, 2]   (identical fits, different weights)
min-norm (pseudo-inverse) solution: [0.6 0.6 1.2] | ⟂ to null vector? True
ridge: XᵀX + 0.1 I invertible? det = 6.321


## Part E — Case study: balancing chemical equations

In [10]:
def balance(elements, reactants, products):
    """Each species is a dict element -> count. Returns smallest positive integer coefficients."""
    species = reactants + products
    M = sp.Matrix([[ (1 if j < len(reactants) else -1) * s.get(e, 0) for j, s in enumerate(species)]
                   for e in elements])
    ns = M.nullspace()
    assert len(ns) == 1, f"nullity {len(ns)}: not a unique reaction"
    v = ns[0]
    v = v * sp.ilcm(*[term.q for term in v])          # clear denominators
    v = v / sp.igcd(*[int(term) for term in v])
    if v[0] < 0: v = -v
    return M, list(v)

M, c = balance(["C", "H", "O"], [{"C": 1, "H": 4}, {"O": 2}], [{"C": 1, "O": 2}, {"H": 2, "O": 1}])
print("CH4 + O2 -> CO2 + H2O  composition matrix:"); sp.pprint(M); print("coefficients:", c)

M, c = balance(["C", "H", "O"], [{"C": 3, "H": 8}, {"O": 2}], [{"C": 1, "O": 2}, {"H": 2, "O": 1}])
print("C3H8 + O2 -> CO2 + H2O:", c)

M, c = balance(["C", "H", "O"], [{"C": 2, "H": 6}, {"O": 2}], [{"C": 1, "O": 2}, {"H": 2, "O": 1}])
print("C2H6 + O2 -> CO2 + H2O:", c)

M, c = balance(["K", "Mn", "O", "H", "Cl"],
               [{"K": 1, "Mn": 1, "O": 4}, {"H": 1, "Cl": 1}],
               [{"K": 1, "Cl": 1}, {"Mn": 1, "Cl": 2}, {"H": 2, "O": 1}, {"Cl": 2}])
print("KMnO4 + HCl -> KCl + MnCl2 + H2O + Cl2:", c, "| rank =", M.rank(), "of", M.shape[1], "species")

CH4 + O2 -> CO2 + H2O  composition matrix:
⎡1  0  -1  0 ⎤
⎢            ⎥
⎢4  0  0   -2⎥
⎢            ⎥
⎣0  2  -2  -1⎦
coefficients: [1, 2, 1, 2]
C3H8 + O2 -> CO2 + H2O: [1, 5, 3, 4]
C2H6 + O2 -> CO2 + H2O: [2, 7, 4, 6]
KMnO4 + HCl -> KCl + MnCl2 + H2O + Cl2: [2, 16, 2, 2, 8, 5] | rank = 5 of 6 species


## Part F — Case study: Hamming(7,4) over GF(2)

In [11]:
H = np.array([[1, 0, 1, 0, 1, 0, 1],
              [0, 1, 1, 0, 0, 1, 1],
              [0, 0, 0, 1, 1, 1, 1]])          # column j is j written in binary (LSB on top)
# Null space of H over GF(2): brute force all 2^7 words
words = np.array([[(k >> i) & 1 for i in range(7)] for k in range(128)])
codewords = words[(words @ H.T % 2 == 0).all(axis=1)]
print("number of codewords (vectors in N(H) over GF(2)):", len(codewords), "= 2^4 -> dimension 4 = 7 - rank 3")

def encode(d):                               # data bits go to positions 3,5,6,7
    c = np.zeros(7, int)
    c[[2, 4, 5, 6]] = d
    c[0] = (c[2] + c[4] + c[6]) % 2          # p1 covers positions 1,3,5,7
    c[1] = (c[2] + c[5] + c[6]) % 2          # p2 covers positions 2,3,6,7
    c[3] = (c[4] + c[5] + c[6]) % 2          # p4 covers positions 4,5,6,7
    return c

def decode(r):
    s = H @ r % 2
    pos = s[0] + 2 * s[1] + 4 * s[2]         # syndrome read as a binary number
    r = r.copy()
    if pos: r[pos - 1] ^= 1
    return s, pos, r[[2, 4, 5, 6]]

c = encode(np.array([1, 0, 1, 1]))
print("data 1011 -> codeword", c, "| H c mod 2 =", H @ c % 2)
r = c.copy(); r[4] ^= 1                      # channel flips bit 5
s, pos, d = decode(r)
print("received", r, "| syndrome", s, "-> error at position", pos, "| decoded data", d)
r2 = c.copy(); r2[[1, 5]] ^= 1               # two errors: Hamming(7,4) cannot fix this
print("two flips ->", decode(r2), "(miscorrects: only single errors are guaranteed)")
print("min weight of a non-zero codeword (min distance):", codewords[1:].sum(axis=1).min())

number of codewords (vectors in N(H) over GF(2)): 16 = 2^4 -> dimension 4 = 7 - rank 3
data 1011 -> codeword [0 1 1 0 0 1 1] | H c mod 2 = [0 0 0]
received [0 1 1 0 1 1 1] | syndrome [1 0 1] -> error at position 5 | decoded data [1 0 1 1]
two flips -> (array([0, 0, 1]), np.int64(4), array([1, 0, 0, 1])) (miscorrects: only single errors are guaranteed)
min weight of a non-zero codeword (min distance): 3


In [12]:
# Practice-problem check: received word 1110000
s, pos, d = decode(np.array([1, 1, 1, 0, 0, 0, 0]))
print("syndrome", s, "position", pos, "data", d)

syndrome [0 0 0] position 0 data [1 0 0 0]


## Part G — Case study: conservation of flow (Kirchhoff's current law)

In [13]:
# Nodes A,B,C,D; directed edges e1:A→B, e2:B→C, e3:C→A, e4:C→D, e5:D→A
edges = [("A", "B"), ("B", "C"), ("C", "A"), ("C", "D"), ("D", "A")]
nodes = ["A", "B", "C", "D"]
M = sp.zeros(4, 5)
for j, (u, v) in enumerate(edges):
    M[nodes.index(u), j] = -1    # flow leaves u
    M[nodes.index(v), j] = 1     # flow enters v
sp.pprint(M)
print("rank =", M.rank(), "| nullity =", len(M.nullspace()), "= E - V + 1 =", 5 - 4 + 1)
print("special solutions (circulations):", [list(v) for v in M.nullspace()])
f = sp.Matrix([3, 3, 1, 2, 2])
print("is f =", list(f), "a valid circulation? M f =", list(M * f))

⎡-1  0   1   0   1 ⎤
⎢                  ⎥
⎢1   -1  0   0   0 ⎥
⎢                  ⎥
⎢0   1   -1  -1  0 ⎥
⎢                  ⎥
⎣0   0   0   1   -1⎦
rank = 3 | nullity = 2 = E - V + 1 = 2
special solutions (circulations): [[1, 1, 1, 0, 0], [1, 1, 0, 1, 1]]
is f = [3, 3, 1, 2, 2] a valid circulation? M f = [0, 0, 0, 0]


## Part H — Case study: multicollinearity, VIF and the dummy-variable trap

In [14]:
rng = np.random.default_rng(42)
n = 500
x1 = rng.normal(size=n); x2 = rng.normal(size=n); x4 = rng.normal(size=n)
x3 = x1 + x2 + 0.05 * rng.normal(size=n)      # nearly redundant
df = pd.DataFrame({"x1": x1, "x2": x2, "x3": x3, "x4": x4})

def vif(df):
    out = {}
    for col in df.columns:
        others = df.drop(columns=col)
        r2 = LinearRegression().fit(others, df[col]).score(others, df[col])
        out[col] = 1 / (1 - r2)
    return pd.Series(out)

v = vif(df)
print(v.round(1))
Xc = df.values - df.values.mean(axis=0)
sv = np.linalg.svd(Xc, compute_uv=False)
print("singular values:", sv.round(3), "| condition number:", round(sv[0] / sv[-1], 1))
print("right singular vector of smallest σ:", np.linalg.svd(Xc)[2][-1].round(3))
df_drop = df.drop(columns="x3")
print("after dropping x3:\n", vif(df_drop).round(3))

x1    368.4
x2    413.8
x3    774.3
x4      1.0
dtype: float64
singular values: [38.09  23.252 21.683  0.645] | condition number: 59.0
right singular vector of smallest σ: [-0.577 -0.577  0.578  0.   ]
after dropping x3:
 x1    1.003
x2    1.001
x4    1.004
dtype: float64


In [15]:
# Exact redundancy: x3 = x1 + x2 exactly -> nullity 1, VIF infinite
Xe = np.c_[x1, x2, x1 + x2, x4]
print("rank =", np.linalg.matrix_rank(Xe), "| nullity =", Xe.shape[1] - np.linalg.matrix_rank(Xe))
print("null vector:", null_space(Xe).ravel().round(4), " ∝ (1, 1, -1, 0)")

rank = 3 | nullity = 1
null vector: [-0.5774 -0.5774  0.5774  0.    ]  ∝ (1, 1, -1, 0)


In [16]:
# Dummy-variable trap: intercept + one-hot of ALL k categories
city = rng.choice(["Bengaluru", "Delhi", "Mumbai"], size=12)
D = pd.get_dummies(city, dtype=float)
X_trap = np.c_[np.ones(len(city)), D.values]            # columns: 1, Bengaluru, Delhi, Mumbai
print("columns: intercept,", list(D.columns))
print("rank =", np.linalg.matrix_rank(X_trap), "of", X_trap.shape[1])
print("null vector (exact):", [list(v) for v in sp.Matrix(X_trap.astype(int)).nullspace()])
X_ok = np.c_[np.ones(len(city)), pd.get_dummies(city, drop_first=True, dtype=float).values]
print("with drop_first=True: rank =", np.linalg.matrix_rank(X_ok), "of", X_ok.shape[1])

columns: intercept, ['Bengaluru', 'Delhi', 'Mumbai']
rank = 3 of 4
null vector (exact): [[-1, 1, 1, 1]]
with drop_first=True: rank = 3 of 3


## Part I — Checks for the practice problems

In [17]:
M_ = sp.Matrix
print("P8 :", [list(v) for v in M_([[1, -1, 2], [-2, 2, -4]]).nullspace()])
print("P9 :", list(M_([[1, 1, -1], [0, 1, 1]]) * M_([2, -1, 1])))
P10 = M_([[1, 2, 1, 4], [2, 4, 3, 11]])
print("P10: RREF[A|b] =", P10.row_join(M_([3, 8])).rref()[0].tolist(), "| N(A) =", [list(v) for v in P10.nullspace()])
P11 = M_([[1, 0, 2, -1], [0, 1, -1, 3], [1, 1, 1, 2]])
print("P11: rank", P11.rank(), "| N(A) =", [list(v) for v in P11.nullspace()])
print("P14: N(A1ᵀ) =", [list(v) for v in A1.T.nullspace()])
print("P17: C2H6 + O2 -> CO2 + H2O:", balance(["C", "H", "O"], [{"C": 2, "H": 6}, {"O": 2}], [{"C": 1, "O": 2}, {"H": 2, "O": 1}])[1])
c = encode(np.array([0, 1, 1, 0])); r = c.copy(); r[2] ^= 1
print("P18: data 0110 -> codeword", c, "| flip bit 3 ->", r, "syndrome", H @ r % 2, "->", decode(r)[1])
E = [("A", "B"), ("B", "C"), ("C", "A"), ("A", "C")]; Mi = sp.zeros(3, 4)
for j, (u, v) in enumerate(E):
    Mi["ABC".index(u), j] = -1; Mi["ABC".index(v), j] = 1
print("P20: rank", Mi.rank(), "| circulations", [list(v) for v in Mi.nullspace()])
D = M_([[1, 1, 0, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 1, 0], [1, 0, 0, 0, 1], [1, 1, 0, 0, 0]])
print("P21: rank", D.rank(), "of 5 | null vector", [list(v) for v in D.nullspace()])
Xq = M_([[1, 2], [0, 1], [3, 1], [2, 2]]); Xq = Xq.row_join(2 * Xq[:, 0] - Xq[:, 1])
print("P24: N(X) =", [list(v) for v in Xq.nullspace()], "| X(1,1,1) =", list(Xq * M_([1, 1, 1])), "| X(3,0,0) =", list(Xq * M_([3, 0, 0])))

P8 : [[1, 1, 0], [-2, 0, 1]]
P9 : [0, 0]
P10: RREF[A|b] = [[1, 2, 0, 1, 1], [0, 0, 1, 3, 2]] | N(A) = [[-2, 1, 0, 0], [-1, 0, -3, 1]]
P11: rank 2 | N(A) = [[-2, 1, 1, 0], [1, -3, 0, 1]]
P14: N(A1ᵀ) = [[-1, -1, 1]]
P17: C2H6 + O2 -> CO2 + H2O: [2, 7, 4, 6]
P18: data 0110 -> codeword [1 1 0 0 1 1 0] | flip bit 3 -> [1 1 1 0 1 1 0] syndrome [1 1 0] -> 3
P20: rank 2 | circulations [[1, 1, 1, 0], [-1, -1, 0, 1]]
P21: rank 4 of 5 | null vector [[-1, 1, 1, 1, 1]]
P24: N(X) = [[-2, 1, 1]] | X(1,1,1) = [3, 0, 9, 6] | X(3,0,0) = [3, 0, 9, 6]
